In [1]:
import neural_layer as nn
import numpy as np

In [2]:
fc = nn.fc(3,2)

In [3]:
fc.weights = np.array([[1,2,3], [4,5,6]])
fc.biases = np.array([[1],[1]])

In [4]:
x = np.array([[1],[2],[3]])
x_vec = np.array([1,2,3])

In [5]:
x_vec

array([1, 2, 3])

In [6]:
x_vec

array([1, 2, 3])

In [7]:
fc(x)

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 3 is different from 1)

In [ ]:
len(x_vec.shape)

1

In [ ]:
import numpy as np
import torch
from activations import ReLU, Sigmoid, Tanh, Softmax


def compare(name, numpy_out, torch_out, tol=1e-7):
    if np.allclose(numpy_out, torch_out, atol=tol):
        print(f"[PASS] {name}")
    else:
        print(f"[FAIL] {name}")
        print("NumPy:", numpy_out)
        print("Torch:", torch_out)


# =========================
# ReLU Tests
# =========================

def test_relu():
    print("\nTesting ReLU")

    for i in range(4):
        x_np = np.random.randn(5, 3)
        x_torch = torch.tensor(x_np, requires_grad=True)

        relu_np = ReLU()
        out_np = relu_np.forward(x_np)

        relu_torch = torch.nn.ReLU()
        out_torch = relu_torch(x_torch)

        compare("ReLU Forward", out_np, out_torch.detach().numpy())

        delta = np.random.randn(*out_np.shape)
        out_torch.backward(torch.tensor(delta))

        grad_np = relu_np.backward(delta)
        grad_torch = x_torch.grad.numpy()

        compare("ReLU Backward", grad_np, grad_torch)

        x_torch.grad.zero_()


# =========================
# Sigmoid Tests
# =========================

def test_sigmoid():
    print("\nTesting Sigmoid")

    for i in range(4):
        x_np = np.random.randn(4, 4)
        x_torch = torch.tensor(x_np, requires_grad=True)

        sig_np = Sigmoid()
        out_np = sig_np.forward(x_np)

        sig_torch = torch.nn.Sigmoid()
        out_torch = sig_torch(x_torch)

        compare("Sigmoid Forward", out_np, out_torch.detach().numpy())

        delta = np.random.randn(*out_np.shape)
        out_torch.backward(torch.tensor(delta))

        grad_np = sig_np.backward(delta)
        grad_torch = x_torch.grad.numpy()

        compare("Sigmoid Backward", grad_np, grad_torch)

        x_torch.grad.zero_()


# =========================
# Tanh Tests
# =========================

def test_tanh():
    print("\nTesting Tanh")

    for i in range(4):
        x_np = np.random.randn(6, 2)
        x_torch = torch.tensor(x_np, requires_grad=True)

        tanh_np = Tanh()
        out_np = tanh_np.forward(x_np)

        tanh_torch = torch.nn.Tanh()
        out_torch = tanh_torch(x_torch)

        compare("Tanh Forward", out_np, out_torch.detach().numpy())

        delta = np.random.randn(*out_np.shape)
        out_torch.backward(torch.tensor(delta))

        grad_np = tanh_np.backward(delta)
        grad_torch = x_torch.grad.numpy()

        compare("Tanh Backward", grad_np, grad_torch)

        x_torch.grad.zero_()


# =========================
# Softmax Tests (column vector)
# =========================

def test_softmax():
    print("\nTesting Softmax")

    for i in range(4):
        x_np = np.random.randn(5, 1)
        x_torch = torch.tensor(x_np.reshape(1, -1), requires_grad=True)

        sm_np = Softmax()
        out_np = sm_np.forward(x_np)

        sm_torch = torch.nn.Softmax(dim=1)
        out_torch = sm_torch(x_torch)

        compare("Softmax Forward", out_np.flatten(), out_torch.detach().numpy().flatten())

        delta = np.random.randn(5, 1)

        out_torch.backward(torch.tensor(delta.reshape(1, -1)))

        grad_np = sm_np.backward(delta)
        grad_torch = x_torch.grad.numpy().reshape(-1, 1)

        compare("Softmax Backward", grad_np, grad_torch)

        x_torch.grad.zero_()


# =========================
# Run All Tests
# =========================

if __name__ == "__main__":
    np.random.seed(0)
    torch.manual_seed(0)

    test_relu()
    test_sigmoid()
    test_tanh()
    test_softmax()

C:\Users\Nagasai\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(



Testing ReLU
[PASS] ReLU Forward
[PASS] ReLU Backward
[PASS] ReLU Forward
[PASS] ReLU Backward
[PASS] ReLU Forward
[PASS] ReLU Backward
[PASS] ReLU Forward
[PASS] ReLU Backward

Testing Sigmoid
[PASS] Sigmoid Forward
[PASS] Sigmoid Backward
[PASS] Sigmoid Forward
[PASS] Sigmoid Backward
[PASS] Sigmoid Forward
[PASS] Sigmoid Backward
[PASS] Sigmoid Forward
[PASS] Sigmoid Backward

Testing Tanh
[PASS] Tanh Forward
[PASS] Tanh Backward
[PASS] Tanh Forward
[PASS] Tanh Backward
[PASS] Tanh Forward
[PASS] Tanh Backward
[PASS] Tanh Forward
[PASS] Tanh Backward

Testing Softmax
[PASS] Softmax Forward
[PASS] Softmax Backward
[PASS] Softmax Forward
[PASS] Softmax Backward
[PASS] Softmax Forward
[PASS] Softmax Backward
[PASS] Softmax Forward
[PASS] Softmax Backward


In [ ]:
import numpy as np
import torch
from neural_layer import fc

torch.set_default_dtype(torch.float64)


def compare(name, numpy_out, torch_out, tol=1e-7):
    if np.allclose(numpy_out, torch_out, atol=tol):
        print(f"[PASS] {name}")
    else:
        print(f"[FAIL] {name}")
        print("Max diff:", np.max(np.abs(numpy_out - torch_out)))


def test_fc():
    print("\nTesting Fully Connected Layer")

    for i in range(4):

        in_dim = 5
        out_dim = 3

        x_np = np.random.randn(in_dim, 1)
        x_torch = torch.tensor(x_np.T, requires_grad=True)

        fc_np = fc(in_dim, out_dim)
        fc_torch = torch.nn.Linear(in_dim, out_dim, bias=True).double()

        # Copy parameters
        fc_torch.weight.data = torch.tensor(fc_np.weights)
        fc_torch.bias.data = torch.tensor(fc_np.biases.flatten())

        # ========= Forward =========
        out_np = fc_np.forward(x_np)
        out_torch = fc_torch(x_torch)

        compare("FC Forward",
                out_np.flatten(),
                out_torch.detach().numpy().flatten())

        # ========= Backward =========
        delta = np.random.randn(out_dim, 1)

        out_torch.backward(torch.tensor(delta.T))

        fc_np.zero_grad()
        grad_input_np = fc_np.backward(delta)

        compare("FC Grad Input",
                grad_input_np,
                x_torch.grad.numpy().T)

        compare("FC Grad Weight",
                fc_np.grad_W,
                fc_torch.weight.grad.numpy())

        compare("FC Grad Bias",
        fc_np.grad_b.flatten(),
        fc_torch.bias.grad.numpy())

        # Reset grads
        x_torch.grad.zero_()
        fc_torch.weight.grad.zero_()
        fc_torch.bias.grad.zero_()


if __name__ == "__main__":
    np.random.seed(0)
    torch.manual_seed(0)
    test_fc()


Testing Fully Connected Layer
[PASS] FC Forward
[PASS] FC Grad Input
[PASS] FC Grad Weight
[PASS] FC Grad Bias
[PASS] FC Forward
[PASS] FC Grad Input
[PASS] FC Grad Weight
[PASS] FC Grad Bias
[PASS] FC Forward
[PASS] FC Grad Input
[PASS] FC Grad Weight
[PASS] FC Grad Bias
[PASS] FC Forward
[PASS] FC Grad Input
[PASS] FC Grad Weight
[PASS] FC Grad Bias


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

from objective_functions import CrossEntropy, MSE

TOL = 1e-7
NUM_TESTS = 10
C = 7  # number of classes / dimensions

np.random.seed(42)
torch.manual_seed(42)


# -------------------------------------------------
# Utility
# -------------------------------------------------

def softmax_np(x):
    x = x - np.max(x)
    exp = np.exp(x)
    return exp / np.sum(exp)


def check_close(name, error):
    if error < TOL:
        print(f"[PASS] {name} | error={error:.3e}")
    else:
        print(f"[FAIL] {name} | error={error:.3e}")


# =================================================
# CROSS ENTROPY TESTS
# =================================================

print("\n===== CROSS ENTROPY TESTS =====")

for test_id in range(NUM_TESTS):

    # Random logits
    logits_np = np.random.randn(C, 1)
    logits_torch = torch.tensor(
        logits_np, dtype=torch.float64, requires_grad=True
    )

    # Softmax probabilities
    probs_np = softmax_np(logits_np)
    probs_torch = F.softmax(logits_torch, dim=0)

    # Random class target
    target_class = np.random.randint(C)

    y_true_np = np.zeros((C, 1))
    y_true_np[target_class] = 1

    y_true_torch = torch.tensor([target_class])

    # ---- Your implementation ----
    ce = CrossEntropy()
    loss_np = ce.forward(probs_np, y_true_np)
    grad_np = ce.backward()

    # ---- Torch reference (probability form) ----
    # Manual CE on probabilities (matches your impl exactly)
    loss_torch = -(F.one_hot(y_true_torch, C)
                   .reshape(C,1)
                   * torch.log(probs_torch)).sum()

    # Backward wrt probabilities
    loss_torch.backward()

    grad_torch = logits_torch.grad  # This is dL/dz, not dL/dp (so we don't use it)

    # Instead compute analytical dL/dp:
    grad_torch_prob = - (F.one_hot(y_true_torch, C)
                         .reshape(C,1)
                         / probs_torch).detach().numpy()

    # ---- Errors ----
    forward_error = abs(loss_np - loss_torch.item())
    backward_error = np.linalg.norm(grad_np - grad_torch_prob)

    check_close(f"CE Forward Test {test_id+1}", forward_error)
    check_close(f"CE Backward Test {test_id+1}", backward_error)


# =================================================
# MSE TESTS
# =================================================

print("\n===== MSE TESTS =====")

for test_id in range(NUM_TESTS):

    y_pred_np = np.random.randn(C, 1)
    y_true_np = np.random.randn(C, 1)

    y_pred_torch = torch.tensor(
        y_pred_np, dtype=torch.float64, requires_grad=True
    )
    y_true_torch = torch.tensor(
        y_true_np, dtype=torch.float64
    )

    # ---- Your implementation ----
    mse = MSE()
    loss_np = mse.forward(y_pred_np, y_true_np)
    grad_np = mse.backward()

    # ---- Torch reference ----
    loss_torch = F.mse_loss(
        y_pred_torch, y_true_torch, reduction='sum'
    )

    loss_torch.backward()

    grad_torch = y_pred_torch.grad.detach().numpy()

    # ---- Errors ----
    forward_error = abs(loss_np - loss_torch.item())
    backward_error = np.linalg.norm(grad_np - grad_torch)

    check_close(f"MSE Forward Test {test_id+1}", forward_error)
    check_close(f"MSE Backward Test {test_id+1}", backward_error)


===== CROSS ENTROPY TESTS =====
[PASS] CE Forward Test 1 | error=0.000e+00
[PASS] CE Backward Test 1 | error=0.000e+00
[PASS] CE Forward Test 2 | error=0.000e+00
[PASS] CE Backward Test 2 | error=0.000e+00
[PASS] CE Forward Test 3 | error=0.000e+00
[PASS] CE Backward Test 3 | error=0.000e+00
[PASS] CE Forward Test 4 | error=0.000e+00
[PASS] CE Backward Test 4 | error=0.000e+00
[PASS] CE Forward Test 5 | error=0.000e+00
[PASS] CE Backward Test 5 | error=0.000e+00
[PASS] CE Forward Test 6 | error=0.000e+00
[PASS] CE Backward Test 6 | error=0.000e+00
[PASS] CE Forward Test 7 | error=0.000e+00
[PASS] CE Backward Test 7 | error=0.000e+00
[PASS] CE Forward Test 8 | error=0.000e+00
[PASS] CE Backward Test 8 | error=0.000e+00
[PASS] CE Forward Test 9 | error=0.000e+00
[PASS] CE Backward Test 9 | error=0.000e+00
[PASS] CE Forward Test 10 | error=0.000e+00
[PASS] CE Backward Test 10 | error=0.000e+00

===== MSE TESTS =====
[PASS] MSE Forward Test 1 | error=3.553e-15
[PASS] MSE Backward Test 1 |

In [ ]:
x = np.zeros(3)
x + 3

array([3., 3., 3.])

In [ ]:
def one_hot(y):
    out = np.zeros((10,1))
    out[y] = 1
    return out

In [ ]:
x = [1,2,3,4]
one_hot(x)

array([[0.],
       [1.],
       [1.],
       [1.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.]])